# Pipeline SFT — FOL (NL → premises-FOL)

`src/models/fol_model/` + `src/data/fol_dataset.py`. Dữ liệu: `premises_nl` / `premises_fol` (bỏ mẫu lệch độ dài). Hub: prefix **`fol-`**.

- Có thể trỏ `FOL_SFT_PROCESSED_DIR` tới `data/processed/fol_sft` (export lọc) hoặc dùng chung `logic_sft`.
- Sau train: `fol_eval_*` trong `experiment_log.json` (exact match JSON).
- Notebook: dependency + `%run -i` các stage.
- **Colab / Kaggle:** full repo có `src/` (clone vào `/content` thường đủ); nếu vẫn lỗi: `import os; os.environ["LOGIC_PROJECT_ROOT"]="/content/.../Logic_Based_Educational_Queries_Project"` rồi chạy lại ô bootstrap.


## 0. Dependency (chạy khi cần)

In [4]:
# %pip install -q transformers datasets peft "trl>=0.16.0" accelerate bitsandbytes scikit-learn huggingface_hub gdown python-dotenv
# %pip install -q trl

## 1. Secrets (Kaggle) / token HF

In [5]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    HF_Token = UserSecretsClient().get_secret("HF_Token")
except Exception:
    HF_Token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or ""

## 2. Bootstrap path cho `services` + `fol_model`

In [6]:
# import os, sys, shlex
# from pathlib import Path
# from IPython import get_ipython

# _SKIP = frozenset(
#     {".git", "__pycache__", ".venv", "venv", "node_modules", "site-packages", ".cache"}
# )


# def _roots():
#     r, seen = [], set()

#     def add(p):
#         try:
#             q = Path(p).expanduser().resolve()
#         except OSError:
#             return
#         if q not in seen:
#             seen.add(q)
#             r.append(q)

#     for k in ("LOGIC_PROJECT_ROOT", "EXACT_PROJECT_ROOT"):
#         v = os.environ.get(k, "").strip()
#         if v:
#             add(v)
#     ip = get_ipython()
#     nb = (ip.user_ns if ip else {}).get("__vsc_ipynb_file__")
#     if nb:
#         q = Path(nb).resolve()
#         add(q.parent)
#         add(q.parent.parent)
#     here = Path.cwd().resolve()
#     add(here)
#     for p in here.parents:
#         add(p)

#     ki = Path("/kaggle/input")
#     if ki.is_dir():
#         try:
#             for a in ki.iterdir():
#                 if not a.is_dir():
#                     continue
#                 add(a)
#                 try:
#                     for b in a.iterdir():
#                         if b.is_dir():
#                             add(b)
#                 except OSError:
#                     pass
#         except OSError:
#             pass

#     def scan_deep(root: Path, max_depth: int) -> None:
#         if not root.is_dir():
#             return
#         stack = [(root.resolve(), 0)]
#         done: set[Path] = set()
#         while stack:
#             d, depth = stack.pop()
#             try:
#                 rd = d.resolve()
#             except OSError:
#                 continue
#             if rd in done:
#                 continue
#             done.add(rd)
#             if (rd / "src" / "services").is_dir():
#                 add(rd)
#             if depth >= max_depth:
#                 continue
#             try:
#                 for ch in d.iterdir():
#                     if (
#                         ch.is_dir()
#                         and ch.name not in _SKIP
#                         and not ch.name.startswith(".")
#                     ):
#                         stack.append((ch, depth + 1))
#             except (OSError, PermissionError):
#                 pass

#     scan_deep(Path("/content"), 14)
#     scan_deep(Path("/content/drive/MyDrive"), 5)
#     return r


# def _repo_root():
#     nest = "Logic_Based_Educational_Queries_Project"
#     for base in _roots():
#         for b in (base, base / nest):
#             if (b / "src" / "services").is_dir():
#                 return b.resolve()
#     raise FileNotFoundError(
#         "Không thấy src/services. Colab: chạy ô trước: "
#         "import os; os.environ['LOGIC_PROJECT_ROOT']='/content/.../Logic_Based_Educational_Queries_Project' "
#         "rồi chạy lại ô này (hoặc git clone full repo vào /content)."
#     )


# R = _repo_root()
# B = R / "notebooks" / "project_bootstrap.py"
# if B.is_file():
#     get_ipython().run_line_magic("run", "-i " + shlex.quote(str(B)))
# else:
#     sys.path.insert(0, str(R / "src"))
#     from services.config import load_dotenv_for_logic

#     print("inline PYTHONPATH:", R / "src", "| .env:", load_dotenv_for_logic())

## 3. Cấu hình FOL — chỉnh tại đây

In [7]:
from services.config_fol import FolSFTConfig

cfg = FolSFTConfig.from_env()
print("Config OK", cfg.model_name)
print("HF repo (push):", cfg.resolved_hf_repo())

ModuleNotFoundError: No module named 'services'

## 3b. (Tuỳ chọn) Export `fol_sft/*.csv` đã lọc
Chạy trong một ô riêng nếu muốn tách khỏi `logic_sft`:

```python
from pathlib import Path
from data.fol_dataset import export_filtered_fol_csvs

root = Path(cfg.app_dir).resolve()
export_filtered_fol_csvs(root / "data/processed/logic_sft", root / "data/processed/fol_sft")
```

In [ ]:
from pathlib import Path
from data.fol_dataset import export_filtered_fol_csvs

root = Path(cfg.app_dir).resolve()
export_filtered_fol_csvs(root / "data/processed/logic_sft", root / "data/processed/fol_sft")

## 4. Tải dữ liệu thô (Drive) — theo `LOGIC_DATA_SOURCE`
Bỏ qua nếu đã có JSON trong `data/raw/`.

In [ ]:
# from services.config import LogicSFTConfig

# if LogicSFTConfig.from_env().data_source == "drive":
#     get_ipython().run_line_magic("run", "-i fol_model_stage_drive.py")
# else:
#     print("DATA_SOURCE=local — bỏ qua tải Drive.")

## 5. Fine-tune FOL (LoRA SFT)

In [ ]:
%run -i fol_model_stage_train.py

## 6. Push Hugging Face (tùy chọn)
Đặt `FOL_PUSH_TO_HUB=true` trong `.env` và có `HF_Token`.

In [ ]:
%run -i fol_model_stage_push.py